In [1]:
import os
import json
from getpass import getpass

import requests
import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
def jprint(value):
    print(json.dumps(value, indent=2, default=str))


def load_tiny_jeopardy():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json"
    return requests.get(url, timeout=30).json()


def load_jeopardy_1k():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/intro-workshop/main/data/jeopardy_1k.json"
    return requests.get(url, timeout=30).json()

In [5]:
def recreate_question_generative_collection(client):
    collection_name = "QuestionGenerative"

    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)

    return client.collections.create(
        name=collection_name,
        vector_config=wvc.config.Configure.Vectors.text2vec_openai(
            model="text-embedding-3-small",
        ),
        generative_config=wvc.config.Configure.Generative.openai(
            model="gpt-4o-mini",
        ),
        properties=[
            wvc.config.Property(
                name="question",
                data_type=wvc.config.DataType.TEXT,
            ),
            wvc.config.Property(
                name="answer",
                data_type=wvc.config.DataType.TEXT,
            ),
            wvc.config.Property(
                name="category",
                data_type=wvc.config.DataType.TEXT,
            ),
        ],
    )

In [6]:
def import_questions(collection, data):
    objects = []

    for item in data:
        objects.append(
            {
                "question": item["Question"],
                "answer": item["Answer"],
                "category": item["Category"],
            }
        )

    result = collection.data.insert_many(objects)

    if result.errors:
        print("Import errors:")
        jprint(result.errors)
    else:
        print("Import completed.")

    return result

In [7]:
questions = recreate_question_generative_collection(client)

data = load_tiny_jeopardy()

print("Records:", len(data))
jprint(data[0])

import_questions(questions, data)

count = questions.aggregate.over_all(total_count=True)

print("Objects in collection:", count.total_count)

Records: 10
{
  "Category": "SCIENCE",
  "Question": "This organ removes excess glucose from the blood & stores it as glycogen",
  "Answer": "Liver"
}
Import completed.
Objects in collection: 10


In [9]:
response = questions.query.near_text(
    query="palnets",
    limit=2,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print(obj.properties)
    print("-" * 80)

Distance: 0.7981168627738953
{'answer': 'the atmosphere', 'question': 'Changes in the tropospheric layer of this are what gives us weather', 'category': 'SCIENCE'}
--------------------------------------------------------------------------------
Distance: 0.8442221283912659
{'answer': 'Liver', 'question': 'This organ removes excess glucose from the blood & stores it as glycogen', 'category': 'SCIENCE'}
--------------------------------------------------------------------------------


In [10]:
prompt = "Tell me a short story about this answer: {answer}"

response = questions.generate.near_text(
    query="animals",
    single_prompt=prompt,
    limit=2,
    return_properties=["question", "answer", "category"],
)

for obj in response.objects:
    print("Question:", obj.properties["question"])
    print("Answer:", obj.properties["answer"])
    print("Generated:", obj.generated)
    print("-" * 80)

Question: It's the only living mammal in the order Proboseidea
Answer: Elephant
Generated: In the heart of a lush jungle, where the trees whispered secrets and the rivers hummed lullabies, there lived a gentle elephant named Elara. Unlike the other elephants who roamed in her herd, Elara was known for her curious spirit and a heart full of wonder.

One warm afternoon, as the sun cast golden beams through the canopy, Elara decided to venture farther than she ever had before. She had heard stories from the older elephants about a magical glade hidden deep within the jungle, a place where the colors of the flowers danced in hues more vibrant than any rainbow.

With each step, she marveled at the beauty around her—the shimmering butterflies that floated like living jewels, the soft rustle of leaves that sang in the breeze, and the comforting sounds of distant animal calls. But as the sun climbed higher, a thick mist rolled in, twisting the familiar paths into a maze. Elara started to feel 

In [11]:
response = questions.generate.near_text(
    query="animals",
    grouped_task="Which of these categories would a zoologist be most interested in? Explain briefly.",
    limit=10,
    return_properties=["category"],
)

print(response.generated)

A zoologist would be most interested in the category "ANIMALS." This is because zoologists specialize in the study of animals, including their behavior, physiology, classification, and ecology. The questions and answers related to animals, such as elephants, antelopes, and snakes, provide insights into various species, their characteristics, and their roles in ecosystems, which are central to zoological research and knowledge. The other category, "SCIENCE," while relevant, encompasses a broader range of scientific disciplines that do not specifically focus on animal life.


In [12]:
response = questions.generate.near_text(
    query="science and biology",
    single_prompt="Explain this answer in one simple sentence: {answer}",
    limit=3,
    return_properties=["question", "answer", "category"],
)

for obj in response.objects:
    print("Question:", obj.properties["question"])
    print("Answer:", obj.properties["answer"])
    print("Generated:", obj.generated)
    print("-" * 80)

Question: In 1953 Watson & Crick built a model of the molecular structure of this, the gene-carrying substance
Answer: DNA
Generated: DNA is the molecule that carries genetic information in living organisms.
--------------------------------------------------------------------------------
Question: 2000 news: the Gunnison sage grouse isn't just another northern sage grouse, but a new one of this classification
Answer: species
Generated: A species is a group of organisms that can interbreed and produce fertile offspring.
--------------------------------------------------------------------------------
Question: This organ removes excess glucose from the blood & stores it as glycogen
Answer: Liver
Generated: The liver is a vital organ that processes nutrients, detoxifies harmful substances, and produces important proteins for blood clotting and metabolism.
--------------------------------------------------------------------------------


In [13]:
questions = recreate_question_generative_collection(client)

data_1k = load_jeopardy_1k()

print("Records:", len(data_1k))
jprint(data_1k[0])

import_questions(questions, data_1k)

count = questions.aggregate.over_all(total_count=True)

print("Objects in collection:", count.total_count)

Records: 1000
{
  "Air Date": "2006-11-08",
  "Round": "Double Jeopardy!",
  "Value": 800,
  "Category": "AMERICAN HISTORY",
  "Question": "Abraham Lincoln died across the street from this theatre on April 15, 1865",
  "Answer": "Ford's Theatre (the Ford Theatre accepted)"
}
Import completed.
Objects in collection: 1000


In [14]:
response = questions.generate.near_text(
    query="space and planets",
    single_prompt="Explain the answer '{answer}' in the context of this question: {question}",
    limit=3,
    return_properties=["question", "answer", "category"],
)

for obj in response.objects:
    print("Category:", obj.properties["category"])
    print("Question:", obj.properties["question"])
    print("Answer:", obj.properties["answer"])
    print("Generated:", obj.generated)
    print("-" * 80)

Category: SCIENCE & NATURE
Question: In March 1966 the USSR's Venera 3 became the first space probe to physically touch another planet, this one
Answer: Venus
Generated: The answer 'Venus' refers to the fact that the Venera 3, a Soviet space probe, was designed to explore the planet Venus. In March 1966, the probe successfully made contact with the surface of Venus, marking a significant achievement as it became the first human-made object to reach and physically touch another planet. This event was historic in the context of space exploration, as it provided valuable data on the atmosphere and surface conditions of Venus, paving the way for future missions to the planet.
--------------------------------------------------------------------------------
Category: SCIENCE
Question: This 2-letter-named moon of Jupiter is the most volcanically active body in the solar system
Answer: Io
Generated: The answer "Io" refers to one of Jupiter's major moons. It is known for being the most volcanic

In [15]:
response = questions.generate.near_text(
    query="history and politics",
    grouped_task="Summarize what common topic connects these Jeopardy questions. Answer in 3 short bullet points.",
    limit=5,
    return_properties=["question", "answer", "category"],
)

print(response.generated)

- All questions pertain to significant figures and events in U.S. history.
- They focus on presidential actions, decisions, or notable attributes.
- Each question highlights important political or social changes in American history.


In [16]:
collections = client.collections.list_all()

for name in collections:
    print(name)

Article
ClipImageCloud
HardwareArticle
Question
QuestionGenerative
QuestionHybrid
QuestionVector


In [17]:
client.close()